In [ ]:
# ===============================
# SQL Queries
# ===============================

# Author: Shelly Resurreccion
# Purpose: This script contains SQL queries to analyze the accident dataset and extract insights based on specified questions.

---
### 1.0 Set Up

In [20]:
# ===============================
# Imports:
# ===============================
# --- Standard library ---
import sys
import os

# --- Data handling ---
import sqlite3
import pandas as pd

In [21]:
# ===============================
# Functions:
# ===============================
# N/A

In [22]:
# ===============================
# Load Data:
# ===============================
# Notes: The dataset was encoded in Latin-1 rather than UTF-8, so the encoding was explicitly specified when loading the CSV.
df = pd.read_csv(
    "../data/US_Accidents_March23.csv",
    encoding="latin1"
)

conn = sqlite3.connect(":memory:")
US_Accidents = df.to_sql("US_Accidents", conn, index=False, if_exists="replace")

---
### Question 1: Timezones

Write a query that denotes the time difference for any given timezone and UTC (i.e. MST is -7). You can ignore any rows with missing Timezone values.

In [23]:
query = """
SELECT DISTINCT
    Timezone,
    CASE
        WHEN LOWER(Timezone) LIKE '%eastern%'  THEN -5
        WHEN LOWER(Timezone) LIKE '%central%'  THEN -6
        WHEN LOWER(Timezone) LIKE '%mountain%' THEN -7
        WHEN LOWER(Timezone) LIKE '%pacific%'  THEN -8
    END AS utc_offset_hours
FROM US_Accidents
WHERE Timezone IS NOT NULL
ORDER BY utc_offset_hours;
"""

pd.read_sql(query, conn)

,Timezone,utc_offset_hours
0,pacific,-8
1,Pacific,-8
2,us/pacific,-8
3,US/Pacific,-8
4,mountain,-7
5,Mountain,-7
6,us/mountain,-7
7,US/Mountain,-7
8,central,-6
9,Central,-6


---
### Question 2: Temperature Range

Write queries related to the Temperature_Range(F) column:

    - i. Convert the temperatures into Celsius for your stakeholders. Use the average from the range. to get C from F, use the following formula 𝐶 = (𝐹 − 32) * 5/9
    - ii. Calculate the median temperature (celsius) for all the accidents in a new query.
    - iii. Impute the missing data

In [24]:
# 2.i Convert the temperatures into Celsius

## Step 1: Parse Temperature_Range(F) into min and max temperatures (min_temp_f, max_temp_f)
## Step 2: Calculate average temperature in Fahrenheit (avg_temp_f)
## Step 3: Convert average temperature to Celsius (avg_temp_c)

query = """
WITH temps AS (
    SELECT
        "Temperature_Range(F)" AS temp_range_f,

        CAST(
            TRIM(SUBSTR("Temperature_Range(F)", 1, INSTR("Temperature_Range(F)", '-') - 1))
            AS REAL
        ) AS min_temp_f,

        CAST(
            TRIM(SUBSTR("Temperature_Range(F)", INSTR("Temperature_Range(F)", '-') + 1))
            AS REAL
        ) AS max_temp_f
    FROM US_Accidents
    WHERE "Temperature_Range(F)" IS NOT NULL
)

SELECT
    temp_range_f,
    min_temp_f,
    max_temp_f,
    (min_temp_f + max_temp_f) / 2.0 AS avg_temp_f,
    ((min_temp_f + max_temp_f) / 2.0 - 32) * 5.0 / 9.0 AS avg_temp_c
FROM temps;
"""

pd.read_sql(query, conn)

,temp_range_f,min_temp_f,max_temp_f,avg_temp_f,avg_temp_c
0,31.0 - 35.0,31.0,35.0,33.0,0.555556
1,30.0 - 34.0,30.0,34.0,32.0,0.000000
2,29.0 - 33.0,29.0,33.0,31.0,-0.555556
3,59.0 - 63.0,59.0,63.0,61.0,16.111111
4,55.0 - 59.0,55.0,59.0,57.0,13.888889
...,...,...,...,...,...
240454,70.0 - 74.0,70.0,74.0,72.0,22.222222
240455,72.0 - 76.0,72.0,76.0,74.0,23.333333
240456,31.0 - 35.0,31.0,35.0,33.0,0.555556
240457,41.0 - 45.0,41.0,45.0,43.0,6.111111


In [25]:
# 2.ii Calculate the median temperature (celsius) for all the accidents in a new query.

## Step 1: Parse the temperature range
## Step 2: Convert to average Celsius (same as before)
## Step 3: Sort temperatures
## Step 4: Pick the middle value(s)
## Step 5: Average them if needed (even number of rows)

query = """
WITH temps AS (
    SELECT
        (
            (
                CAST(TRIM(SUBSTR("Temperature_Range(F)", 1, INSTR("Temperature_Range(F)", '-') - 1)) AS REAL)
              + CAST(TRIM(SUBSTR("Temperature_Range(F)", INSTR("Temperature_Range(F)", '-') + 1)) AS REAL)
            ) / 2.0 - 32
        ) * 5.0 / 9.0 AS avg_temp_c
    FROM US_Accidents
    WHERE "Temperature_Range(F)" IS NOT NULL
),

ordered AS (
    SELECT
        avg_temp_c,
        ROW_NUMBER() OVER (ORDER BY avg_temp_c) AS rn,
        COUNT(*) OVER () AS total_rows
    FROM temps
)

SELECT
    AVG(avg_temp_c) AS median_temp_c
FROM ordered
WHERE rn IN (
    (total_rows + 1) / 2,
    (total_rows + 2) / 2
);
"""

pd.read_sql(query, conn)

,median_temp_c
0,9.444444


In [26]:
# 2.iii Impute the missing data

## Step 1: Review current statistics of the data
query = """
WITH temps_no_imputation AS (
    SELECT
        "Temperature_Range(F)" AS temp_range_f,

        CAST(
            TRIM(SUBSTR("Temperature_Range(F)", 1, INSTR("Temperature_Range(F)", '-') - 1))
            AS REAL
        ) AS min_temp_f,

        CAST(
            TRIM(SUBSTR("Temperature_Range(F)", INSTR("Temperature_Range(F)", '-') + 1))
            AS REAL
        ) AS max_temp_f
    FROM US_Accidents
    WHERE "Temperature_Range(F)" IS NOT NULL
),

temps_no_imputation_c AS (
    SELECT
        temp_range_f,
        min_temp_f,
        max_temp_f,
        (min_temp_f + max_temp_f) / 2.0 AS avg_temp_f,
        ((min_temp_f + max_temp_f) / 2.0 - 32) * 5.0 / 9.0 AS avg_temp_c
    FROM temps_no_imputation
)

SELECT
    COUNT(*) AS total_rows,
    COUNT(temp_range_f) AS non_missing,
    COUNT(*) - COUNT(temp_range_f) AS missing_count,
    AVG(avg_temp_c) AS mean_c,
    AVG(avg_temp_f) AS mean_f,
    MIN(min_temp_f) AS min_f,
    MAX(max_temp_f) AS max_f
FROM temps_no_imputation_c;
"""

pd.read_sql(query, conn)

,total_rows,non_missing,missing_count,mean_c,mean_f,min_f,max_f
0,240459,240459,0,8.878219,47.980795,0.0,209.0


In [27]:
## Step 2: Determine which imputation method to use

### Option 1: Constant Imputation – Replace missing values with a fixed value (e.g., 20).
query = """
WITH temps_constant_imputation AS (
    SELECT
        "Temperature_Range(F)" AS temp_range_f,

        -- Constant imputation for min_temp_f: if NULL or no value, set to 0
        CAST(
            TRIM(
                COALESCE(
                    SUBSTR("Temperature_Range(F)", 1, INSTR("Temperature_Range(F)", '-') - 1),
                    '0'
                )
            ) AS REAL
        ) AS min_temp_f,

        -- Constant imputation for max_temp_f: if NULL or no value, set to 0
        CAST(
            TRIM(
                COALESCE(
                    SUBSTR("Temperature_Range(F)", INSTR("Temperature_Range(F)", '-') + 1),
                    '0'
                )
            ) AS REAL
        ) AS max_temp_f

    FROM US_Accidents
),

temps_constant_imputation_c AS (
    SELECT
        temp_range_f,
        min_temp_f,
        max_temp_f,
        (min_temp_f + max_temp_f) / 2.0 AS avg_temp_f,
        ((min_temp_f + max_temp_f) / 2.0 - 32) * 5.0 / 9.0 AS avg_temp_c
    FROM temps_constant_imputation 
)

SELECT
    COUNT(*) AS total_rows,
    COUNT(temp_range_f) AS non_missing,
    COUNT(*) - COUNT(temp_range_f) AS missing_count,
    AVG(avg_temp_c) AS mean_c,
    AVG(avg_temp_f) AS mean_f,
    MIN(min_temp_f) AS min_f,
    MAX(max_temp_f) AS max_f
FROM temps_constant_imputation_c;
"""

pd.read_sql(query, conn)

,total_rows,non_missing,missing_count,mean_c,mean_f,min_f,max_f
0,246633,240459,6174,8.210936,46.779685,0.0,209.0


In [28]:
### Option 2: Mean Imputation – Replace missing values with the average of the column.
query = """
WITH mean_vals AS (
    -- Calculate the mean for min and max temperatures
    SELECT
        AVG(CAST(TRIM(SUBSTR("Temperature_Range(F)", 1, INSTR("Temperature_Range(F)", '-') - 1)) AS REAL)) AS mean_min,
        AVG(CAST(TRIM(SUBSTR("Temperature_Range(F)", INSTR("Temperature_Range(F)", '-') + 1)) AS REAL)) AS mean_max
    FROM US_Accidents
    WHERE "Temperature_Range(F)" IS NOT NULL
),

temps_mean_imputation AS (
    SELECT
        a."Temperature_Range(F)" AS temp_range_f,

        -- Mean imputation for min_temp_f
        CAST(
            TRIM(
                COALESCE(
                    SUBSTR(a."Temperature_Range(F)", 1, INSTR(a."Temperature_Range(F)", '-') - 1),
                    mv.mean_min
                )
            ) AS REAL
        ) AS min_temp_f,

        -- Mean imputation for max_temp_f
        CAST(
            TRIM(
                COALESCE(
                    SUBSTR(a."Temperature_Range(F)", INSTR(a."Temperature_Range(F)", '-') + 1),
                    mv.mean_max
                )
            ) AS REAL
        ) AS max_temp_f

    FROM US_Accidents a
    CROSS JOIN mean_vals mv
),

temps_mean_imputation_c AS (
    SELECT
        temp_range_f,
        min_temp_f,
        max_temp_f,
        (min_temp_f + max_temp_f) / 2.0 AS avg_temp_f,
        ((min_temp_f + max_temp_f) / 2.0 - 32) * 5.0 / 9.0 AS avg_temp_c
    FROM temps_mean_imputation
)

SELECT
    COUNT(*) AS total_rows,
    COUNT(temp_range_f) AS non_missing,
    COUNT(*) - COUNT(temp_range_f) AS missing_count,
    AVG(avg_temp_c) AS mean_c,
    AVG(avg_temp_f) AS mean_f,
    MIN(min_temp_f) AS min_f,
    MAX(max_temp_f) AS max_f
FROM temps_mean_imputation_c;
"""

pd.read_sql(query, conn)

,total_rows,non_missing,missing_count,mean_c,mean_f,min_f,max_f
0,246633,240459,6174,8.878219,47.980795,0.0,209.0


---
### Question 3: Total Road Length Impacted

Write queries related to the Distance(mi) column:

    - i. Calculate the total road length impacted by accidents for each city.
    - ii. Rank cities based on their total road length impacted. Include a column identifying rank 1-10.
    - iii. Return only the top 10 cities, along with their total road length and rank.

In [29]:
# 3.i Calculate the total road length impacted by accidents for each city.

query = """
SELECT
    City,
    COUNT(ID) AS total_accidents,
    SUM("Distance(mi)") AS total_road_length_impacted,
    COUNT(ID) * 1.0 / SUM(COUNT(ID)) OVER () AS pct_of_total_accidents
FROM US_Accidents
GROUP BY City
ORDER BY total_accidents DESC;
"""

pd.read_sql(query, conn)

,City,total_accidents,total_road_length_impacted,pct_of_total_accidents
0,Los Angeles,5880,7661.325,0.023841
1,Miami,5238,2868.977,0.021238
2,Dallas,3261,2724.503,0.013222
3,Atlanta,3182,3472.670,0.012902
4,San Diego,2759,2142.034,0.011187
...,...,...,...,...
6640,Zenia,1,0.325,0.000004
6641,Zieglerville,1,0.270,0.000004
6642,Zortman,1,0.459,0.000004
6643,Zuni,1,0.234,0.000004


In [30]:
# 3.ii Rank cities based on their total road length impacted. Include a column identifying rank 1-10.

## Step 1: Make and review the bins for ranking (rank 1 - 10)
#  OPTION 1
query = """
WITH total_road_length AS (
    SELECT
        City,
        COUNT(ID) AS total_accidents,
        SUM("Distance(mi)") AS total_road_length_impacted
    FROM US_Accidents
    GROUP BY City
),

stats AS (
    SELECT 
        MIN(total_road_length_impacted) AS min_val,
        MAX(total_road_length_impacted) AS max_val,
        (MAX(total_road_length_impacted) - MIN(total_road_length_impacted)) / 10.0 AS bin_width
    FROM total_road_length
)

SELECT
    ROUND(s.min_val + i * s.bin_width, 3) AS bin_start,
    CASE WHEN i = 9 THEN ROUND(s.max_val, 3)
         ELSE ROUND(s.min_val + (i+1) * s.bin_width, 3)
    END AS bin_end,
    i + 1 AS bin_rank
FROM stats s,
     (SELECT 0 AS i UNION ALL SELECT 1 UNION ALL SELECT 2 UNION ALL SELECT 3 UNION ALL 
      SELECT 4 UNION ALL SELECT 5 UNION ALL SELECT 6 UNION ALL SELECT 7 UNION ALL 
      SELECT 8 UNION ALL SELECT 9) bins
ORDER BY bin_rank;
"""

pd.read_sql(query, conn)

,bin_start,bin_end,bin_rank
0,0.003,766.135,1
1,766.135,1532.267,2
2,1532.267,2298.400,3
3,2298.400,3064.532,4
4,3064.532,3830.664,5
5,3830.664,4596.796,6
6,4596.796,5362.928,7
7,5362.928,6129.061,8
8,6129.061,6895.193,9
9,6895.193,7661.325,10


In [31]:
#  OPTION 2
query = """
WITH city_totals AS (
    SELECT
        City,
        SUM("Distance(mi)") AS total_road_length_impacted
    FROM US_Accidents
    GROUP BY City
),
stats AS (
    SELECT
        MIN(total_road_length_impacted) AS min_val,
        MAX(total_road_length_impacted) AS max_val,
        (MAX(total_road_length_impacted) - MIN(total_road_length_impacted)) / 10.0 AS bin_width
    FROM city_totals
)
SELECT
    ROUND(min_val + (bin_rank - 1) * bin_width, 3) AS bin_start,
    ROUND(
        CASE
            WHEN bin_rank = 10 THEN max_val
            ELSE min_val + bin_rank * bin_width
        END,
        3
    ) AS bin_end,
    bin_rank
FROM stats
CROSS JOIN (
    SELECT ROW_NUMBER() OVER () AS bin_rank
    FROM US_Accidents
    LIMIT 10
) bins
ORDER BY bin_rank;
"""

pd.read_sql(query, conn)

,bin_start,bin_end,bin_rank
0,0.003,766.135,1
1,766.135,1532.267,2
2,1532.267,2298.400,3
3,2298.400,3064.532,4
4,3064.532,3830.664,5
5,3830.664,4596.796,6
6,4596.796,5362.928,7
7,5362.928,6129.061,8
8,6129.061,6895.193,9
9,6895.193,7661.325,10


In [32]:
## Step 2: Assign those bins back to the cities in the data
#  OPTION 1
query = """
WITH total_road_length AS (
    SELECT
        City,
        COUNT(ID) AS total_accidents,
        SUM("Distance(mi)") AS total_road_length_impacted
    FROM US_Accidents
    GROUP BY City
),
stats AS (
    SELECT 
        MIN(total_road_length_impacted) AS min_val,
        MAX(total_road_length_impacted) AS max_val,
        (MAX(total_road_length_impacted) - MIN(total_road_length_impacted)) / 10.0 AS bin_width
    FROM total_road_length
)
SELECT
    t.City,
    t.total_accidents,
    t.total_road_length_impacted,
    -- Compute bin index from 0 to 9
    CASE
        WHEN FLOOR((t.total_road_length_impacted - s.min_val) / s.bin_width) > 9 THEN 10
        ELSE FLOOR((t.total_road_length_impacted - s.min_val) / s.bin_width) + 1
    END AS bin_rank,
    CONCAT(
        -- Bin start
        ROUND(
            CASE 
                WHEN FLOOR((t.total_road_length_impacted - s.min_val) / s.bin_width) > 9 
                THEN s.min_val + 9 * s.bin_width
                ELSE s.min_val + FLOOR((t.total_road_length_impacted - s.min_val) / s.bin_width) * s.bin_width
            END
        , 3),
        '-',
        -- Bin end
        ROUND(
            CASE 
                WHEN FLOOR((t.total_road_length_impacted - s.min_val) / s.bin_width) > 9 
                THEN s.max_val
                ELSE s.min_val + (FLOOR((t.total_road_length_impacted - s.min_val) / s.bin_width) + 1) * s.bin_width
            END
        , 3)
    ) AS road_length_bin
FROM total_road_length t
CROSS JOIN stats s
ORDER BY bin_rank DESC, t.total_road_length_impacted DESC;
"""   

pd.read_sql(query, conn)

,City,total_accidents,total_road_length_impacted,bin_rank,road_length_bin
0,Los Angeles,5880,7661.325,10.0,6895.193-7661.325
1,Atlanta,3182,3472.670,5.0,3064.532-3830.664
2,Miami,5238,2868.977,4.0,2298.4-3064.532
3,Dallas,3261,2724.503,4.0,2298.4-3064.532
4,Phoenix,1752,2313.209,4.0,2298.4-3064.532
...,...,...,...,...,...
6640,Old Mill Creek,1,0.006,1.0,0.003-766.135
6641,Round Lake Heights,1,0.006,1.0,0.003-766.135
6642,Fordyce,1,0.005,1.0,0.003-766.135
6643,Sun Lakes,1,0.005,1.0,0.003-766.135


In [ ]:
# OPTION 2
query = """
WITH city_totals AS (
    SELECT
        City,
        COUNT(ID) AS total_accidents,
        SUM("Distance(mi)") AS total_road_length_impacted
    FROM US_Accidents
    GROUP BY City
),
stats AS (
    SELECT
        MIN(total_road_length_impacted) AS min_val,
        MAX(total_road_length_impacted) AS max_val,
        (MAX(total_road_length_impacted) - MIN(total_road_length_impacted)) / 10.0 AS bin_width
    FROM city_totals
),
binned AS (
    SELECT
        c.*,
        s.min_val,
        s.max_val,
        s.bin_width,
        CASE
            WHEN FLOOR((c.total_road_length_impacted - s.min_val) / s.bin_width) > 9
                THEN 9
            ELSE FLOOR((c.total_road_length_impacted - s.min_val) / s.bin_width)
        END AS bin_index
    FROM city_totals c
    CROSS JOIN stats s
)
SELECT
    City,
    total_accidents,
    total_road_length_impacted,
    bin_index + 1 AS bin_rank,
    ROUND(min_val + bin_index * bin_width, 3)
        || '-' ||
    ROUND(
        CASE
            WHEN bin_index = 9 THEN max_val
            ELSE min_val + (bin_index + 1) * bin_width
        END,
        3
    ) AS road_length_bin
FROM binned
ORDER BY bin_rank DESC, total_road_length_impacted DESC;
"""

pd.read_sql(query, conn)

,City,total_accidents,total_road_length_impacted,bin_rank,road_length_bin
0,Los Angeles,5880,7661.325,10.0,6895.193-7661.325
1,Atlanta,3182,3472.670,5.0,3064.532-3830.664
2,Miami,5238,2868.977,4.0,2298.4-3064.532
3,Dallas,3261,2724.503,4.0,2298.4-3064.532
4,Phoenix,1752,2313.209,4.0,2298.4-3064.532
...,...,...,...,...,...
6640,Old Mill Creek,1,0.006,1.0,0.003-766.135
6641,Round Lake Heights,1,0.006,1.0,0.003-766.135
6642,Fordyce,1,0.005,1.0,0.003-766.135
6643,Sun Lakes,1,0.005,1.0,0.003-766.135


In [ ]:
# 3.iii Return only the top 10 cities, along with their total road length and rank.

# OPTION 1
query = """
WITH total_road_length AS (
    SELECT
        City,
        COUNT(ID) AS total_accidents,
        SUM("Distance(mi)") AS total_road_length_impacted
    FROM US_Accidents
    GROUP BY City
),
stats AS (
    SELECT 
        MIN(total_road_length_impacted) AS min_val,
        MAX(total_road_length_impacted) AS max_val,
        (MAX(total_road_length_impacted) - MIN(total_road_length_impacted)) / 10.0 AS bin_width
    FROM total_road_length
)
SELECT
    t.City,
    t.total_accidents,
    t.total_road_length_impacted,
    -- Compute bin index from 0 to 9
    CASE
        WHEN FLOOR((t.total_road_length_impacted - s.min_val) / s.bin_width) > 9 THEN 10
        ELSE FLOOR((t.total_road_length_impacted - s.min_val) / s.bin_width) + 1
    END AS bin_rank,
    CONCAT(
        -- Bin start
        ROUND(
            CASE 
                WHEN FLOOR((t.total_road_length_impacted - s.min_val) / s.bin_width) > 9 
                THEN s.min_val + 9 * s.bin_width
                ELSE s.min_val + FLOOR((t.total_road_length_impacted - s.min_val) / s.bin_width) * s.bin_width
            END
        , 3),
        '-',
        -- Bin end
        ROUND(
            CASE 
                WHEN FLOOR((t.total_road_length_impacted - s.min_val) / s.bin_width) > 9 
                THEN s.max_val
                ELSE s.min_val + (FLOOR((t.total_road_length_impacted - s.min_val) / s.bin_width) + 1) * s.bin_width
            END
        , 3)
    ) AS road_length_bin
FROM total_road_length t
CROSS JOIN stats s
ORDER BY bin_rank DESC, t.total_road_length_impacted DESC
LIMIT 10;
"""   

pd.read_sql(query, conn)

,City,total_accidents,total_road_length_impacted,bin_rank,road_length_bin
0,Los Angeles,5880,7661.325,10.0,6895.193-7661.325
1,Atlanta,3182,3472.670,5.0,3064.532-3830.664
2,Miami,5238,2868.977,4.0,2298.4-3064.532
3,Dallas,3261,2724.503,4.0,2298.4-3064.532
4,Phoenix,1752,2313.209,4.0,2298.4-3064.532
5,San Diego,2759,2142.034,3.0,1532.267-2298.4
6,Truckee,205,2031.721,3.0,1532.267-2298.4
7,Flagstaff,259,1717.482,3.0,1532.267-2298.4
8,Corona,963,1618.023,3.0,1532.267-2298.4
9,Minneapolis,1555,1605.242,3.0,1532.267-2298.4


In [ ]:
# OPTION 2
query = """
WITH city_totals AS (
    SELECT
        City,
        COUNT(ID) AS total_accidents,
        SUM("Distance(mi)") AS total_road_length_impacted
    FROM US_Accidents
    GROUP BY City
),
stats AS (
    SELECT
        MIN(total_road_length_impacted) AS min_val,
        MAX(total_road_length_impacted) AS max_val,
        (MAX(total_road_length_impacted) - MIN(total_road_length_impacted)) / 10.0 AS bin_width
    FROM city_totals
),
binned AS (
    SELECT
        c.*,
        s.min_val,
        s.max_val,
        s.bin_width,
        CASE
            WHEN FLOOR((c.total_road_length_impacted - s.min_val) / s.bin_width) > 9
                THEN 9
            ELSE FLOOR((c.total_road_length_impacted - s.min_val) / s.bin_width)
        END AS bin_index
    FROM city_totals c
    CROSS JOIN stats s
)
SELECT
    City,
    total_accidents,
    total_road_length_impacted,
    bin_index + 1 AS bin_rank,
    ROUND(min_val + bin_index * bin_width, 3)
        || '-' ||
    ROUND(
        CASE
            WHEN bin_index = 9 THEN max_val
            ELSE min_val + (bin_index + 1) * bin_width
        END,
        3
    ) AS road_length_bin
FROM binned
ORDER BY bin_rank DESC, total_road_length_impacted DESC
LIMIT 10;
"""

pd.read_sql(query, conn)